# Kho-Sight — Auto-label remaining frames with your trained model

Runs your fine-tuned role detector over the frames you haven't annotated yet and
produces a `frames + labels` zip you import into Roboflow — so the remaining
annotation becomes 'skim and fix' instead of 'draw from scratch'.

**Why this notebook (not Roboflow's hosted Auto Label):** this workspace's Roboflow
free plan blocks BOTH hosted inference (the Auto Label run failed 401 Unauthorized on
every image) AND weights export (the "Download Weights" button says "upgrade to access").
So the Roboflow-trained v3 model can't be used to label anything on this plan. The fix is
to train the same version 3 dataset in Colab — you own those weights — and label here.

**Do this first:** run `train_khosight_colab.ipynb` (Option A pulls version 3). Its Step 5
saves `khosight_roles_v3.pt` to `MyDrive/khosight/`. On 36 images it finishes fast and,
because it trains at imgsz 1280 (vs Roboflow's ~640), it's actually better on the small
`runner`/`chaser-active` players than the Roboflow model was.

**Then this notebook needs, in `MyDrive/khosight/`:**
- `khosight_roles_v3.pt` — produced by the training notebook above.
- `unlabelled.zip` — a zip of the frames to label. The repo already has `frames/`
  (973) and an `unlabelled.zip`; upload that zip to Drive. Running over all 973 is fine —
  the 36 already in your Roboflow dataset stay untouched when you re-upload with
  "skip duplicates".
- OPTIONAL but recommended: `court_regions.json` (from
  `python scripts/pick_court.py --frames frames --out court_regions.json`) — detections
  outside the court polygon (spectators, benched players) are dropped automatically.

Runtime → T4 GPU makes it fast (~10 min for ~900 frames); CPU works but is slower.

In [ ]:
%pip -q install ultralytics

In [ ]:
# get weights + frames from Drive
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/khosight/khosight_roles_v3.pt /content/best.pt
!unzip -qo /content/drive/MyDrive/khosight/unlabelled.zip -d /content/unlabelled
import glob
# frames may be nested one level deep depending on how the zip was made
IMGS = sorted(glob.glob('/content/unlabelled/**/*.jpg', recursive=True) +
              glob.glob('/content/unlabelled/**/*.png', recursive=True))
print(len(IMGS), 'frames to label')

In [ ]:
# predict and write YOLO-format labels next to a copy of each image,
# dropping detections whose feet are outside the court polygon (if provided)
import json, os, pathlib, shutil
from ultralytics import YOLO

CONF = 0.4   # raise for fewer-but-surer boxes, lower to catch more
MARGIN = 15  # px tolerance so boundary-straddling players are kept

regions = {}
rp = '/content/drive/MyDrive/khosight/court_regions.json'
if os.path.exists(rp):
    regions = json.load(open(rp))
    print('court regions loaded for:', list(regions))
else:
    print('no court_regions.json found - keeping ALL detections')

def point_in_polygon(x, y, poly):
    inside = False
    for i in range(len(poly)):
        x1, y1 = poly[i]; x2, y2 = poly[(i + 1) % len(poly)]
        if (y1 > y) != (y2 > y) and x < (x2 - x1) * (y - y1) / (y2 - y1) + x1:
            inside = not inside
    return inside

def on_court(box, poly):
    fx, fy = (box[0] + box[2]) / 2, box[3]
    return any(point_in_polygon(fx + dx, fy + dy, poly)
               for dx in (-MARGIN, 0, MARGIN) for dy in (-MARGIN, 0, MARGIN))

out_img = pathlib.Path('/content/prelabelled/images'); out_img.mkdir(parents=True, exist_ok=True)
out_lbl = pathlib.Path('/content/prelabelled/labels'); out_lbl.mkdir(parents=True, exist_ok=True)

model = YOLO('/content/best.pt')
print('classes:', model.names)
dropped = 0
for i in range(0, len(IMGS), 64):
    for res in model.predict(IMGS[i:i+64], conf=CONF, verbose=False):
        stem = pathlib.Path(res.path).stem
        poly = regions.get(stem.rsplit('_', 1)[0])
        h, w = res.orig_shape
        lines = []
        for xyxy, xywhn, cls in zip(res.boxes.xyxy.cpu().numpy(),
                                    res.boxes.xywhn.cpu().numpy(),
                                    res.boxes.cls.cpu().numpy()):
            if poly is not None and not on_court(xyxy, poly):
                dropped += 1
                continue
            cx, cy, bw, bh = xywhn
            lines.append(f"{int(cls)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        (out_lbl / f'{stem}.txt').write_text('\n'.join(lines))
        shutil.copy(res.path, out_img / pathlib.Path(res.path).name)
    print(f'{min(i+64, len(IMGS))}/{len(IMGS)}')

names = [model.names[k] for k in sorted(model.names)]
pathlib.Path('/content/prelabelled/data.yaml').write_text(
    'train: images\nval: images\nnc: %d\nnames: %s\n' % (len(names), names))
print(f'done - dropped {dropped} off-court detections')

In [ ]:
# zip and download -> then in Roboflow: Upload Data -> drop the zip.
# Images arrive WITH annotations; correct them in Annotate as usual.
!cd /content && zip -qr prelabelled.zip prelabelled
!cp /content/prelabelled.zip /content/drive/MyDrive/khosight/prelabelled.zip
from google.colab import files
files.download('/content/prelabelled.zip')

## After importing into Roboflow
1. Skim each frame: fix wrong classes, add missed players, delete ghost boxes.
   Expect ~5-10 s per frame instead of 30-40 s.
2. When everything is annotated, generate a new dataset version (YOLOv11 export)
   and rerun `train_khosight_colab.ipynb` for the full-strength model.